## Import packages

In [ ]:
import astropy
from pylab import *
from astropy.io import fits
from astropy.time import Time
from PyAstronomy import pyasl
from astropy import units as u
from astropy.coordinates import SkyCoord, EarthLocation


## Read the FITS file

## 1- In the following line, we will read the observational data from the .fits file given to you

In [ ]:
hdul = fits.open('../../../data/he1158-2313_uncor.fits')      # our star
head = hdul[1].header
data = hdul[1].data
wvl2, flx2 = data['wavelength'],data['norm_flux']             # option 2: this file has norm_flux
# merged orders are not strictly monotonic - crosscorrRV needs sorted wavelengths
import numpy as np
_o = np.argsort(wvl2, kind='stable'); wvl2, flx2 = np.asarray(wvl2,float)[_o], np.asarray(flx2,float)[_o]
data # to check on the file structure

In [ ]:
#wvl1, flx1 = pyasl.read1dFitsSpec("../../../data/HD140283.fits")  # the other reference AF offers

wvl1, flx1 = pyasl.read1dFitsSpec("../../../data/HD122563.fits")   # different format, hence this reader
# checked: this file is already at rest (H-alpha core at 6562.74 A), so no pre-shift is needed.
# HD 122563 is a cool metal-poor GIANT, so a closer physical match to HE 1158-2313 than the
# subgiant HD 140283 - its line strengths should resemble ours more closely.


## Plot the observed spectrum and a comparison spectrum near the Mg b lines to take a look

In [ ]:
fig  = plt.figure(figsize=(12,7))
#plt.plot(wvl2, flx2, 'r.-',label="j0232-3708")
plt.plot(wvl2, flx2, 'r.-',label="HE 1158-2313")

#plt.plot(wvl1, flx1, 'b--',label="HD 122563")
plt.plot(wvl1, flx1, 'b--',label="HD 122563")


# lines below are used to control the tick markers
plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
# lines below are used to add labels
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
# lines below are used to control the x- and y-axis limits. This wavelength range shows the Mg triplet lines.
xlim([5161,5190])
ylim([0.0,1.1])
# line below is used to show the legend
plt.legend(loc="lower right", fontsize=16)
# line below is used to show the figure
plt.show()


In [ ]:
# AF's default window is H-alpha; this uses the Mg b region she suggests instead, because
# H-alpha is the worst line in the spectrum for a velocity (see the markdown above).
#star
ind = np.where((5160.0 < wvl2) & (wvl2< 5200.0))
waveSpectrum = wvl2[ind[0]]
fluxSpectrum = flx2[ind[0]]

#template
indx= np.where((5140.0 < wvl1) & (wvl1< 5220.0))
waveTemplate = wvl1[indx[0]]
fluxTemplate = flx1[indx[0]]


## Plot the spectral region you want to run the cross-correlation on (= Mg b region) to check that everything looks alright

In [ ]:
# Plot the chosen wavelength regime. Recall that the comparison star spectrum is at rest, the observed one isn't (yet).
fig  = plt.figure(figsize=(12,7))
plt.plot(waveSpectrum, fluxSpectrum, 'r.-',label="HE 1158-2313")

#plt.plot(waveTemplate, fluxTemplate, 'b--',label="HD122563")
plt.plot(waveTemplate, fluxTemplate, 'b--',label="HD 122563")

plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
xlim([5160,5200])
ylim([0.0,1.1])
plt.legend(loc="lower right", fontsize=16)
plt.show()


## Carry out the cross-correlation.

In [ ]:

# AF scans -10..+200 km/s, which does NOT contain our star at ~+350 (her comment assumes a
# shift under 3 A; ours is 6 A). Widened; everything else is her call, unchanged.
# Second change to her call: she correlates the RAW fluxes. Both spectra sit near a continuum
# of 1 with absorption dips, so sum(f*t) is LARGEST where the dips fail to overlap - the naive
# CCF has no real peak and argmax lands on the scan edge (verified: it returns rvmin exactly).
# Subtracting the continuum makes the lines the signal, which is what the method assumes.
fluxSpectrum_cs = fluxSpectrum - np.median(fluxSpectrum)
fluxTemplate_cs = fluxTemplate - np.median(fluxTemplate)
rv, cc = pyasl.crosscorrRV(waveSpectrum, fluxSpectrum_cs, waveTemplate, fluxTemplate_cs, -100., 500, 0.1, skipedge=20)



## Find the maximum the of cross-correlation function

In [ ]:
maxind = np.argmax(cc)

## Plot the cross-correlation result

In [ ]:
plt.plot(rv, cc, 'bp-')
plt.plot(rv[maxind], cc[maxind], 'ro')
plt.show()

## Print the value of RV

In [ ]:
print (round(rv[maxind], 2))  # just the plain number  --- USE FOR APPLY_RADVEL_VELOCITY

In [ ]:
print("HE 1158-2313 has a geocentric radial velocity = ", round(rv[maxind], 2), " km/s ")
print()
if rv[maxind] > 0.0:
    print("There is a red-shift with respect to the template")
else:
    print("There is a blue-shift with respect to the template")

print('Is the star receding or approaching us?')

### Save for the other notebooks

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)

# The adopted velocity is the mean over several metal-line windows, not the single window above.
METAL_WINDOWS = {"Mg b 5160-5200": (5160, 5200), "Fe 4400-4700": (4400, 4700),
                 "Fe 5250-5500": (5250, 5500), "Fe 5500-5800": (5500, 5800),
                 "Fe/Cr 4700-5000": (4700, 5000)}
EXCLUDED = {"H-alpha 6560-6575": (6560, 6575)}

def ccf_window(lo, hi):
    m = (wvl2 > lo) & (wvl2 < hi)
    mt = (wvl1 > lo - 20) & (wvl1 < hi + 20)
    r, c = pyasl.crosscorrRV(wvl2[m], flx2[m] - np.median(flx2[m]),
                             wvl1[mt], flx1[mt] - np.median(flx1[mt]),
                             250, 450, 0.1, skipedge=20)
    return float(r[np.argmax(c)])

per_window = {nm: ccf_window(*lh) for nm, lh in METAL_WINDOWS.items()}
for nm, lh in EXCLUDED.items():
    per_window[nm] = ccf_window(*lh)

vals = np.array([per_window[nm] for nm in METAL_WINDOWS])
v_mean, v_sem, v_sc = vals.mean(), vals.std(ddof=1) / np.sqrt(len(vals)), vals.std(ddof=1)
for nm in per_window:
    flag = "   <- EXCLUDED (see above)" if nm in EXCLUDED else ""
    print(f"  {nm:20s} {per_window[nm]:+8.2f} km/s{flag}")
print(f"  {'METAL-LINE MEAN':20s} {v_mean:+8.2f} +/- {v_sem:.2f} km/s  (scatter {v_sc:.2f}, N={len(vals)})")

out = {"v_geo_kms": round(float(v_mean), 2),
       "v_geo_err_kms": round(float(v_sem), 2),
       "window_scatter_kms": round(float(v_sc), 2),
       "per_window_kms": {k: round(v, 2) for k, v in per_window.items()},
       "excluded": list(EXCLUDED),
       "method": "pyasl.crosscorrRV over metal-line windows; AF's notebook, H-alpha window replaced",
       "template": "HD122563.fits (already at rest)",
       "rv_scan_kms": [250.0, 450.0, 0.1], "skipedge": 20,
       "_note": "H-alpha excluded: broad, chromospheric core, non-Gaussian/NLTE profile, and the "
                "most template-sensitive line here. Measured above for comparison only."}
json.dump(out, open("results/part4-2-2_geocentric_rv_HD122563.json", "w"), indent=2)
print()
print(json.dumps(out, indent=2))